# Heston model by Monte Carlo

We implement a Heston model (supposed already calibrated) using a Monte Carlo method. More precisely, we generate many approximate trajectories from the Heston SDE with Euler scheme, and average their respective payoffs to get the final price of the option. The Heston model as given from the exercice sheet writes

\begin{align*}
    & \frac{dS_t}{S_t} = r dt + \sqrt{V_t} dW_t^S \\
    & dV_t = \kappa(\theta - V_t)dt + \eta \sqrt{V_t} dW^V_t \\
    & d\langle W^V, W^S \rangle_t = \rho dt
\end{align*}

In [24]:
import numpy as np
np.random.seed(1)

In [25]:
# Params (from S&P500)
S0 = 110
K = 130
r = 0.05
kappa = 7.2
theta = 0.045
volvol = 0.05
v0 = theta # Initialize the variance as the log term variance

# Correlation of used brownians
rho = 0.1

iterations = 1000000

T = 2 # 2 years 
timestepperyear = 12 # One timestep = One mounth
timesteps = T * timestepperyear
dt = 1/timestepperyear

In [26]:
S_t = np.zeros((timesteps, iterations))
V_t = np.zeros((timesteps, iterations))

V_t[0,:] = v0
S_t[0, :] = S0

for i in range(1, timesteps):
    # We generate multiple random variables to account for the non deterministic part of the equation
    Z1 = np.random.standard_normal(iterations)
    eps = np.random.standard_normal(iterations)
    Z2 = rho*Z1 + np.sqrt(1 - rho**2)*eps

    # calculate V_t[i, :] from V_t[i-1, :] (we update every path in parallel). We use a max for computational safety.
    V_t[i,:] = np.maximum(V_t[i-1,:] + kappa * (theta - V_t[i-1,:])* dt + volvol *  np.sqrt(V_t[i-1,:] * dt) * Z2,0)
    
    # Same for S_t
    S_t[i] = S_t[i-1] * np.exp((r - 0.5*V_t[i-1,:])*dt + np.sqrt(V_t[i-1,:]*dt)*Z1)

payoff = np.maximum(S_t[-1] - K, 0.0)
disc_payoff = np.exp(-r*T)*payoff
price = np.mean(disc_payoff)
stderr = np.std(disc_payoff)/np.sqrt(iterations)
print(f'Price by monte carlo (using {iterations} iterations) : {price} +- {stderr}')

Price by monte carlo (using 1000000 iterations) : 9.585602248472227 +- 0.019830490144432947
